# SENTIRA — Decision-Level Fusion (UC4–UC7)

Loads the saved prediction arrays from the Audio, Video, and EEG branches (each trained and evaluated independently in their own notebooks), aligns them on the 4 subjects shared across all three branches' test sets (Subjects 15, 16, 18, 41), and computes the two fusion rules used in the paper (Eq. 1 — static F1-weighted average; Eq. 2 — per-sample confidence-weighted average) for all seven modality combinations (UC1–UC7).

**Requires:** `uc1_predictions.npz` (Audio), `uc3_predictions_v2.npz` (EEG), `uc2_predictions_shared_subjects.npz` (Video) — all produced by their respective branch notebooks. No model training happens here.

## Cell 1 — Mount Google Drive

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## Cell 2 — Locate Saved Prediction Files (Audio)

In [6]:
from pathlib import Path

# Poori THESIS folder me dhoondein
drive_root = Path("/content/drive/MyDrive/THESIS")
for p in drive_root.rglob("uc1_predictions.npz"):
    print(p)

/content/drive/MyDrive/THESIS/Audio Results/Results/audio/uc1_predictions.npz


## Cell 3 — Locate Saved Prediction Files (EEG)

In [7]:
for p in drive_root.rglob("uc3_predictions_v2.npz"):
    print(p)

/content/drive/MyDrive/THESIS/EEG Results/Results/eeg/uc3_predictions_v2.npz


## Cell 4 — Locate Saved Prediction Files (All Branches)

In [8]:
for p in drive_root.rglob("*predictions*.npz"):
    print(p)

/content/drive/MyDrive/THESIS/Models/Sentira_V12_Training/uc2_predictions_shared_subjects.npz
/content/drive/MyDrive/THESIS/Audio Results/Results/audio/uc1_predictions.npz
/content/drive/MyDrive/THESIS/EEG Results/Results/eeg/uc3_predictions_v2.npz


## Cell 5 — Decision-Level Fusion: Draft Version

> ⚠️ **Note:** This cell is an earlier draft of the fusion computation, superseded by Cell 7 below (which adds confidence-weighted fusion and extra verification checks). It is kept here only because Cell 6's audio-only baseline check depends on the variables it defines. If you re-run this notebook top-to-bottom, both this cell and Cell 7 will run — Cell 7's results are the ones actually reported in the paper.

In [9]:
# ══════════════════════════════════════════════════════════════════════════
# FINAL DECISION-LEVEL FUSION — UC4, UC5, UC6, UC7
# Aligns Audio/EEG/Video predictions on shared subjects {15,16,18,41}
# ══════════════════════════════════════════════════════════════════════════
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, classification_report

SHARED_SUBJECTS = [15, 16, 18, 41]
LABEL_NAMES = ["Happiness", "Sadness", "Angry", "Calmness", "Neutral"]

# ── Adjust these 3 paths to your actual Drive locations ──
AUDIO_NPZ = Path("/content/drive/MyDrive/THESIS/Audio Results/Results/audio/uc1_predictions.npz")
EEG_NPZ   = Path("/content/drive/MyDrive/THESIS/EEG Results/Results/eeg/uc3_predictions_v2.npz")
VIDEO_NPZ = Path("/content/drive/MyDrive/THESIS/Models/Sentira_V12_Training/uc2_predictions_shared_subjects.npz")

def extract_subject_num(key: str) -> int:
    """Pulls the subject number out of any of the three key formats."""
    key_lower = key.lower()
    import re
    m = re.search(r'subject(\d+)', key_lower)
    return int(m.group(1))

def build_ranked_table(sample_keys, y_true, y_probs, shared_subjects):
    """
    Groups samples by (subject, true_label), sorts by original array order
    (preserves each notebook's own trial ordering), assigns a rank within
    each (subject, label) group. Returns dict: (subject,label,rank) -> (probs, true)
    """
    table = {}
    counters = {}
    for i, key in enumerate(sample_keys):
        sub = extract_subject_num(key)
        if sub not in shared_subjects:
            continue
        label = int(y_true[i])
        counter_key = (sub, label)
        rank = counters.get(counter_key, 0)
        counters[counter_key] = rank + 1
        table[(sub, label, rank)] = y_probs[i]
    return table

# ── Load Audio ──
audio = np.load(AUDIO_NPZ, allow_pickle=True)
audio_table = build_ranked_table(audio["sample_keys"], audio["y_true"],
                                  audio["y_probs"], SHARED_SUBJECTS)
print(f"Audio  : {len(audio_table)} samples on shared subjects")

# ── Load EEG — aggregate 4 epochs per trial first ──
eeg = np.load(EEG_NPZ, allow_pickle=True)
eeg_keys, eeg_true, eeg_probs = eeg["sample_keys"], eeg["y_true"], eeg["y_probs"]

import re
from collections import defaultdict
trial_groups = defaultdict(list)   # (subject, trial_num) -> list of row indices
for i, key in enumerate(eeg_keys):
    sub = extract_subject_num(key)
    if sub not in SHARED_SUBJECTS:
        continue
    m = re.search(r'_T(\d+)_E\d+', key)
    trial_num = int(m.group(1))
    trial_groups[(sub, trial_num)].append(i)

eeg_trial_keys, eeg_trial_true, eeg_trial_probs = [], [], []
for (sub, trial_num), idxs in trial_groups.items():
    avg_probs = eeg_probs[idxs].mean(axis=0)          # average the 4 epoch softmaxes
    true_label = int(eeg_true[idxs[0]])                # all 4 epochs share the same label
    eeg_trial_keys.append(f"subject{sub}_trial{trial_num}")
    eeg_trial_true.append(true_label)
    eeg_trial_probs.append(avg_probs)

eeg_trial_true  = np.array(eeg_trial_true)
eeg_trial_probs = np.array(eeg_trial_probs)
eeg_table = build_ranked_table(eeg_trial_keys, eeg_trial_true, eeg_trial_probs, SHARED_SUBJECTS)
print(f"EEG    : {len(eeg_table)} trial-level samples on shared subjects "
      f"(aggregated from {len(eeg_keys)} epochs)")

# ── Load Video (already filtered to shared subjects) ──
video = np.load(VIDEO_NPZ, allow_pickle=True)
video_table = build_ranked_table(video["sample_keys"], video["y_true"],
                                  video["y_probs"], SHARED_SUBJECTS)
print(f"Video  : {len(video_table)} samples on shared subjects")

# ── Find the common (subject, label, rank) keys across all three ──
common_ids = set(audio_table) & set(eeg_table) & set(video_table)
common_ids = sorted(common_ids)
print(f"\n✅ Common aligned trials across ALL 3 modalities: {len(common_ids)}")

if len(common_ids) == 0:
    raise ValueError("No common trials found — check key formats / rank assumption.")

y_true_final   = np.array([audio_table[cid][1] if False else None for cid in common_ids])
# (true label taken from audio_table's stored true value — rebuild properly below)

# Rebuild aligned arrays cleanly
audio_probs_aligned, eeg_probs_aligned, video_probs_aligned, y_true_aligned = [], [], [], []
for (sub, label, rank) in common_ids:
    audio_probs_aligned.append(audio_table[(sub, label, rank)])
    eeg_probs_aligned.append(eeg_table[(sub, label, rank)])
    video_probs_aligned.append(video_table[(sub, label, rank)])
    y_true_aligned.append(label)

audio_probs_aligned = np.array(audio_probs_aligned)
eeg_probs_aligned    = np.array(eeg_probs_aligned)
video_probs_aligned  = np.array(video_probs_aligned)
y_true_aligned       = np.array(y_true_aligned)

print(f"Aligned shapes -> audio:{audio_probs_aligned.shape}  "
      f"eeg:{eeg_probs_aligned.shape}  video:{video_probs_aligned.shape}")

# ── Validated (own-test-set) macro-F1 per modality — used as fusion weights ──
W_AUDIO, W_EEG, W_VIDEO = 0.9750, 0.4109, 0.4437
total_w = W_AUDIO + W_EEG + W_VIDEO
W_AUDIO, W_EEG, W_VIDEO = W_AUDIO/total_w, W_EEG/total_w, W_VIDEO/total_w

def evaluate(y_true, y_pred, name):
    acc = accuracy_score(y_true, y_pred)
    mf1 = f1_score(y_true, y_pred, average="macro")
    wf1 = f1_score(y_true, y_pred, average="weighted")
    kappa = cohen_kappa_score(y_true, y_pred)
    print(f"\n{'='*55}\n{name}\n{'='*55}")
    print(f"  n            : {len(y_true)}")
    print(f"  Accuracy     : {acc*100:.2f}%")
    print(f"  Macro-F1     : {mf1:.4f}")
    print(f"  Weighted-F1  : {wf1:.4f}")
    print(f"  Cohen kappa  : {kappa:.4f}")
    print(classification_report(y_true, y_pred, target_names=LABEL_NAMES,
                                 digits=4, zero_division=0))
    return {"n": int(len(y_true)), "accuracy": round(float(acc),4),
             "macro_f1": round(float(mf1),4), "weighted_f1": round(float(wf1),4),
             "kappa": round(float(kappa),4)}

results = {}

# ── UC4: Audio + Video ──
fused = W_AUDIO*audio_probs_aligned + W_VIDEO*video_probs_aligned
results["UC4_audio_video"] = evaluate(y_true_aligned, fused.argmax(1), "UC4: Audio + Video")

# ── UC5: Audio + EEG ──
fused = W_AUDIO*audio_probs_aligned + W_EEG*eeg_probs_aligned
results["UC5_audio_eeg"] = evaluate(y_true_aligned, fused.argmax(1), "UC5: Audio + EEG")

# ── UC6: Video + EEG ──
fused = W_VIDEO*video_probs_aligned + W_EEG*eeg_probs_aligned
results["UC6_video_eeg"] = evaluate(y_true_aligned, fused.argmax(1), "UC6: Video + EEG")

# ── UC7: Audio + Video + EEG ──
fused = W_AUDIO*audio_probs_aligned + W_VIDEO*video_probs_aligned + W_EEG*eeg_probs_aligned
results["UC7_all_three"] = evaluate(y_true_aligned, fused.argmax(1), "UC7: Audio + Video + EEG")

# ── Save everything for the paper ──
import json
OUT_PATH = Path("/content/drive/MyDrive/THESIS/fusion_results_UC4_UC7.json")
with open(OUT_PATH, "w") as f:
    json.dump(results, f, indent=2)
print(f"\n✅ All fusion results saved: {OUT_PATH}")

print("\n" + "="*70)
print("  IEEE PAPER — FINAL ROADMAP TABLE (fill in from above)")
print("="*70)
for uc, r in results.items():
    print(f"  {uc:<20}: n={r['n']}  Acc={r['accuracy']*100:.2f}%  "
          f"MacroF1={r['macro_f1']:.4f}  Kappa={r['kappa']:.4f}")

Audio  : 400 samples on shared subjects
EEG    : 400 trial-level samples on shared subjects (aggregated from 2400 epochs)
Video  : 400 samples on shared subjects

✅ Common aligned trials across ALL 3 modalities: 400
Aligned shapes -> audio:(400, 5)  eeg:(400, 5)  video:(400, 5)

UC4: Audio + Video
  n            : 400
  Accuracy     : 99.50%
  Macro-F1     : 0.9950
  Weighted-F1  : 0.9950
  Cohen kappa  : 0.9938
              precision    recall  f1-score   support

   Happiness     1.0000    1.0000    1.0000        80
     Sadness     1.0000    0.9750    0.9873        80
       Angry     0.9756    1.0000    0.9877        80
    Calmness     1.0000    1.0000    1.0000        80
     Neutral     1.0000    1.0000    1.0000        80

    accuracy                         0.9950       400
   macro avg     0.9951    0.9950    0.9950       400
weighted avg     0.9951    0.9950    0.9950       400


UC5: Audio + EEG
  n            : 400
  Accuracy     : 99.50%
  Macro-F1     : 0.9950
  Weight

## Cell 6 — Sanity Check: Audio-Alone Baseline Accuracy

In [10]:
audio_only_pred = audio_probs_aligned.argmax(axis=1)
audio_only_acc = (audio_only_pred == y_true_aligned).mean() * 100
print(f"Audio-ALONE accuracy on same 400-subset: {audio_only_acc:.2f}%")

Audio-ALONE accuracy on same 400-subset: 99.50%


## Cell 7 — Decision-Level Fusion: Final Version (with Confidence-Weighted Comparison)

In [3]:
# ══════════════════════════════════════════════════════════════════════════
# FINAL DECISION-LEVEL FUSION — UC4, UC5, UC6, UC7  (with verification + confidence-weighted comparison)
# ══════════════════════════════════════════════════════════════════════════
import numpy as np
import re
from pathlib import Path
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, classification_report

SHARED_SUBJECTS = [15, 16, 18, 41]
LABEL_NAMES = ["Happiness", "Sadness", "Angry", "Calmness", "Neutral"]

AUDIO_NPZ = Path("/content/drive/MyDrive/THESIS/Audio Results/Results/audio/uc1_predictions.npz")
EEG_NPZ   = Path("/content/drive/MyDrive/THESIS/EEG Results/Results/eeg/uc3_predictions_v2.npz")
VIDEO_NPZ = Path("/content/drive/MyDrive/THESIS/Models/Sentira_V12_Training/uc2_predictions_shared_subjects.npz")

def extract_subject_num(key: str) -> int:
    m = re.search(r'subject(\d+)', key.lower())
    return int(m.group(1))

def build_ranked_table(sample_keys, y_true, y_probs, shared_subjects):
    table = {}
    counters = {}
    for i, key in enumerate(sample_keys):
        sub = extract_subject_num(key)
        if sub not in shared_subjects:
            continue
        label = int(y_true[i])
        counter_key = (sub, label)
        rank = counters.get(counter_key, 0)
        counters[counter_key] = rank + 1
        table[(sub, label, rank)] = y_probs[i]
    return table

# ── Load Audio ──
audio = np.load(AUDIO_NPZ, allow_pickle=True)
audio_table = build_ranked_table(audio["sample_keys"], audio["y_true"], audio["y_probs"], SHARED_SUBJECTS)
print(f"Audio  : {len(audio_table)} samples on shared subjects")

# ── Load EEG — aggregate 4 epochs per trial first ──
eeg = np.load(EEG_NPZ, allow_pickle=True)
eeg_keys, eeg_true, eeg_probs = eeg["sample_keys"], eeg["y_true"], eeg["y_probs"]

trial_groups = defaultdict(list)
for i, key in enumerate(eeg_keys):
    sub = extract_subject_num(key)
    if sub not in SHARED_SUBJECTS:
        continue
    m = re.search(r'_T(\d+)_E\d+', key)
    trial_num = int(m.group(1))
    trial_groups[(sub, trial_num)].append(i)

eeg_trial_keys, eeg_trial_true, eeg_trial_probs = [], [], []
for (sub, trial_num), idxs in trial_groups.items():
    avg_probs = eeg_probs[idxs].mean(axis=0)
    true_label = int(eeg_true[idxs[0]])
    eeg_trial_keys.append(f"subject{sub}_trial{trial_num}")
    eeg_trial_true.append(true_label)
    eeg_trial_probs.append(avg_probs)

eeg_trial_true  = np.array(eeg_trial_true)
eeg_trial_probs = np.array(eeg_trial_probs)
eeg_table = build_ranked_table(eeg_trial_keys, eeg_trial_true, eeg_trial_probs, SHARED_SUBJECTS)
print(f"EEG    : {len(eeg_table)} trial-level samples on shared subjects (aggregated from {len(eeg_keys)} epochs)")

# ── Load Video ──
video = np.load(VIDEO_NPZ, allow_pickle=True)
video_table = build_ranked_table(video["sample_keys"], video["y_true"], video["y_probs"], SHARED_SUBJECTS)
print(f"Video  : {len(video_table)} samples on shared subjects")

# ══════════════════════════════════════════════════════════════════════════
# STEP 1: VERIFY PER-(SUBJECT, LABEL) COUNTS MATCH ACROSS ALL THREE MODALITIES
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("  ALIGNMENT VERIFICATION: per-(subject, label) sample counts")
print("="*70)

mismatch_found = False
for sub in SHARED_SUBJECTS:
    for label in range(5):
        a_count = sum(1 for k in audio_table if k[0] == sub and k[1] == label)
        e_count = sum(1 for k in eeg_table if k[0] == sub and k[1] == label)
        v_count = sum(1 for k in video_table if k[0] == sub and k[1] == label)
        status = "OK" if (a_count == e_count == v_count) else "⚠️ MISMATCH"
        print(f"  subject={sub:<3} label={LABEL_NAMES[label]:<10} A={a_count} E={e_count} V={v_count}  {status}")
        if not (a_count == e_count == v_count):
            mismatch_found = True

if mismatch_found:
    raise ValueError("❌ Sample counts do not match across modalities for at least one (subject, label) group.")
else:
    print("\n✅ All (subject, label) group counts match across Audio, EEG, and Video.")

# ── Find common (subject, label, rank) keys ──
common_ids = sorted(set(audio_table) & set(eeg_table) & set(video_table))
print(f"\n✅ Common aligned trials across ALL 3 modalities: {len(common_ids)}")
if len(common_ids) == 0:
    raise ValueError("No common trials found.")

audio_probs_aligned, eeg_probs_aligned, video_probs_aligned, y_true_aligned = [], [], [], []
for (sub, label, rank) in common_ids:
    audio_probs_aligned.append(audio_table[(sub, label, rank)])
    eeg_probs_aligned.append(eeg_table[(sub, label, rank)])
    video_probs_aligned.append(video_table[(sub, label, rank)])
    y_true_aligned.append(label)

audio_probs_aligned = np.array(audio_probs_aligned)
eeg_probs_aligned    = np.array(eeg_probs_aligned)
video_probs_aligned  = np.array(video_probs_aligned)
y_true_aligned       = np.array(y_true_aligned)
print(f"Aligned shapes -> audio:{audio_probs_aligned.shape}  eeg:{eeg_probs_aligned.shape}  video:{video_probs_aligned.shape}")

# ── Static performance-weighted fusion (original method) ──
W_AUDIO, W_EEG, W_VIDEO = 0.9750, 0.4109, 0.4437
total_w = W_AUDIO + W_EEG + W_VIDEO
W_AUDIO, W_EEG, W_VIDEO = W_AUDIO/total_w, W_EEG/total_w, W_VIDEO/total_w

def evaluate(y_true, y_pred, name):
    acc = accuracy_score(y_true, y_pred)
    mf1 = f1_score(y_true, y_pred, average="macro")
    wf1 = f1_score(y_true, y_pred, average="weighted")
    kappa = cohen_kappa_score(y_true, y_pred)
    print(f"\n{'='*55}\n{name}\n{'='*55}")
    print(f"  n={len(y_true)}  Accuracy={acc*100:.2f}%  MacroF1={mf1:.4f}  Kappa={kappa:.4f}")
    return {"n": int(len(y_true)), "accuracy": round(float(acc),4), "macro_f1": round(float(mf1),4),
            "weighted_f1": round(float(wf1),4), "kappa": round(float(kappa),4)}

results = {}
fused = W_AUDIO*audio_probs_aligned + W_VIDEO*video_probs_aligned
results["UC4_audio_video"] = evaluate(y_true_aligned, fused.argmax(1), "UC4: Audio + Video (static)")
fused = W_AUDIO*audio_probs_aligned + W_EEG*eeg_probs_aligned
results["UC5_audio_eeg"] = evaluate(y_true_aligned, fused.argmax(1), "UC5: Audio + EEG (static)")
fused = W_VIDEO*video_probs_aligned + W_EEG*eeg_probs_aligned
results["UC6_video_eeg"] = evaluate(y_true_aligned, fused.argmax(1), "UC6: Video + EEG (static)")
fused = W_AUDIO*audio_probs_aligned + W_VIDEO*video_probs_aligned + W_EEG*eeg_probs_aligned
results["UC7_all_three"] = evaluate(y_true_aligned, fused.argmax(1), "UC7: Audio + Video + EEG (static)")

# ══════════════════════════════════════════════════════════════════════════
# STEP 2: TRUE PER-SAMPLE CONFIDENCE-WEIGHTED FUSION (matches paper's equation)
# ══════════════════════════════════════════════════════════════════════════
def confidence_weighted_fuse(audio_probs, video_probs, eeg_probs, conf_gate=0.3):
    n = len(audio_probs)
    fused_preds = np.zeros(n, dtype=np.int64)
    for i in range(n):
        probs_list = [audio_probs[i], video_probs[i], eeg_probs[i]]
        confs = [p.max() for p in probs_list]
        kept_probs = [p for p, c in zip(probs_list, confs) if c >= conf_gate]
        kept_confs = [c for c in confs if c >= conf_gate]
        if not kept_probs:
            kept_probs, kept_confs = probs_list, confs
        total_conf = sum(kept_confs)
        fused = np.zeros(5)
        for p, c in zip(kept_probs, kept_confs):
            fused += (c / total_conf) * p
        fused_preds[i] = fused.argmax()
    return fused_preds

print("\n" + "="*70)
print("  COMPARISON: Static Performance-Weighted vs. Per-Sample Confidence-Weighted")
print("="*70)

zeros = np.zeros_like(audio_probs_aligned)
configs = {
    "UC4_audio_video": (audio_probs_aligned, video_probs_aligned, zeros),
    "UC5_audio_eeg":   (audio_probs_aligned, zeros, eeg_probs_aligned),
    "UC6_video_eeg":   (zeros, video_probs_aligned, eeg_probs_aligned),
    "UC7_all_three":   (audio_probs_aligned, video_probs_aligned, eeg_probs_aligned),
}

results_confidence = {}
for uc_name, (a, v, e) in configs.items():
    preds = confidence_weighted_fuse(a, v, e)
    acc = (preds == y_true_aligned).mean()
    mf1 = f1_score(y_true_aligned, preds, average="macro")
    kappa = cohen_kappa_score(y_true_aligned, preds)
    results_confidence[uc_name] = {"accuracy": round(float(acc),4), "macro_f1": round(float(mf1),4), "kappa": round(float(kappa),4)}
    static_acc = results[uc_name]["accuracy"]
    print(f"\n  {uc_name}:")
    print(f"    Static (performance-weighted)   : {static_acc*100:.2f}%")
    print(f"    Confidence-weighted (per-sample): {acc*100:.2f}%")
    print(f"    {'✅ MATCH' if abs(static_acc - acc) < 0.01 else '⚠️ DIFFERENT — investigate'}")

import json
with open(Path("/content/drive/MyDrive/THESIS/fusion_results_both_methods.json"), "w") as f:
    json.dump({"static_performance_weighted": results, "confidence_weighted": results_confidence}, f, indent=2)
print("\n✅ Saved both fusion methods for paper comparison.")

Audio  : 400 samples on shared subjects
EEG    : 400 trial-level samples on shared subjects (aggregated from 2400 epochs)
Video  : 400 samples on shared subjects

  ALIGNMENT VERIFICATION: per-(subject, label) sample counts
  subject=15  label=Happiness  A=20 E=20 V=20  OK
  subject=15  label=Sadness    A=20 E=20 V=20  OK
  subject=15  label=Angry      A=20 E=20 V=20  OK
  subject=15  label=Calmness   A=20 E=20 V=20  OK
  subject=15  label=Neutral    A=20 E=20 V=20  OK
  subject=16  label=Happiness  A=20 E=20 V=20  OK
  subject=16  label=Sadness    A=20 E=20 V=20  OK
  subject=16  label=Angry      A=20 E=20 V=20  OK
  subject=16  label=Calmness   A=20 E=20 V=20  OK
  subject=16  label=Neutral    A=20 E=20 V=20  OK
  subject=18  label=Happiness  A=20 E=20 V=20  OK
  subject=18  label=Sadness    A=20 E=20 V=20  OK
  subject=18  label=Angry      A=20 E=20 V=20  OK
  subject=18  label=Calmness   A=20 E=20 V=20  OK
  subject=18  label=Neutral    A=20 E=20 V=20  OK
  subject=41  label=Happin

## Cell 8 — Print Final Fusion Results (UC7 static + confidence-weighted)

In [4]:
print(results["UC7_all_three"])
print(results_confidence)

{'n': 400, 'accuracy': 0.995, 'macro_f1': 0.995, 'weighted_f1': 0.995, 'kappa': 0.9938}
{'UC4_audio_video': {'accuracy': 0.995, 'macro_f1': 0.995, 'kappa': 0.9938}, 'UC5_audio_eeg': {'accuracy': 0.9925, 'macro_f1': 0.9925, 'kappa': 0.9906}, 'UC6_video_eeg': {'accuracy': 0.515, 'macro_f1': 0.4954, 'kappa': 0.3938}, 'UC7_all_three': {'accuracy': 0.995, 'macro_f1': 0.995, 'kappa': 0.9938}}
